In [8]:
#필요한 라이브러리 설치 및 불러우기
import os
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import matplotlib.pyplot as plt
import json

# 더 필요한 라이브러리 추가 -------------
from tqdm import tqdm
import urllib
from urllib.request import Request, urlopen
from PIL import Image
from io import BytesIO
import numpy as np

In [10]:
def get_emergency_info(url, serviceKey):
    params = {
        'serviceKey': serviceKey,
        'pageNo': '1', 'numOfRows': '1000',  # 전체 응급실 수가 500여개 됨. 1000개면 충분
        'format': 'xml'
    }

    response = requests.get(url, params = params)

    # 정상 수행 되었다면 200
    print(response)

    # response xml에서 주요 정보 찾기
    root = ET.fromstring(response.text)
    #target_audio = df_audio.iloc[0]
    data = []

    for item in tqdm(root.findall('.//item')):
        # 필요한 정보 추가
        duty_name = item.findtext('dutyName')
        duty_addr = item.findtext('dutyAddr')
        duty_hayn = item.findtext('dutyHayn')
        duty_hano = item.findtext('dutyHano')
        duty_eryn = item.findtext('dutyEryn') # 응급실 운영 여부
        duty_egk = item.findtext('MKioskTy25') # Emergency gate keeper
        if duty_egk != None:
            duty_egk = duty_egk.strip()
        duty_lon = item.findtext('wgs84Lon') # 경도
        duty_lat = item.findtext('wgs84Lat')

        # 빈 리스트 data에 딕시너리 형태({'칼럼이름':값, ...})로 저장(추가)
        data.append({'병원 이름':duty_name, '병원 주소':duty_addr,
                     '입원실':duty_hayn, '입원실가용여부':duty_hano,
                    '응급실':duty_eryn, 'EGK':duty_egk,
                     '경도':float(duty_lon), '위도':float(duty_lat)})

    # 데이터프레임으로 변환
    df = pd.DataFrame(data)
    #df = df[df['EGK']=='Y'].sort_values(by='거리')
    return df

In [12]:
url = 'http://apis.data.go.kr/B552657/ErmctInfoInqireService/getEgytBassInfoInqire'
serviceKey = 'wiOWpi/Q9kR9/ACtObc8FjQm7P1AzREHFkXBcvXSeI0qavFydFsJTLJX73ocJqkQy4TVdvwooWDLomvtO7u+6g=='     # 여러분의 일반 인증키(Decoding)
emergency_df = get_emergency_info(url, serviceKey)
emergency_df.to_csv('병원 기본 정보.csv')

<Response [200]>


100%|██████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 101872.73it/s]
